# D2 Forestry Management Workflow — GeoPandas

Stand risk and forest management using GeoPandas spatial operations.


In [1]:
from __future__ import annotations

import json, os

from pathlib import Path

from urllib.error import URLError

from urllib.request import Request, urlopen

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

import numpy as np



_ROOT = Path.cwd()
if (_ROOT / "examples" / "notebooks").exists():
    OUTPUT_DIR = _ROOT / "examples" / "notebooks" / "geopandas" / "outputs"
else:
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALLOW_LIVE_API = os.getenv("GEOPROMPT_ALLOW_LIVE_API", "0") == "1"



def fetch_json(url, fallback):

    if not ALLOW_LIVE_API:

        return fallback

    try:

        req = Request(url, headers={"User-Agent": "geoprompt-notebook/2.0"})

        with urlopen(req, timeout=6) as r:

            return json.loads(r.read().decode("utf-8"))

    except (URLError, TimeoutError, ValueError):

        return fallback



def fetch_first_json(urls, validator, fallback):

    for url in urls:

        payload = fetch_json(url, None)

        if payload is not None and validator(payload):

            return payload, url, True

    return fallback, "fallback", False



import geopandas as gpd

from shapely.geometry import Point, Polygon, box

print(f"geopandas {gpd.__version__} ready")


geopandas 1.1.3 ready


## Section A: Pull Data Sources


In [2]:
forest = {"features": [{"id": "fallback-forest"}]}
weather = {"properties": {"forecast": "fallback"}}
forecast = {"hourly": {"temperature_2m": [0.0]}}

forest, forest_src, forest_live = fetch_first_json(
    [
        "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson",
        "https://api.github.com/repos/opengeospatial/geopandas",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("features") or d.get("id")),
    forest,
)
weather, wx_src, wx_live = fetch_first_json(
    [
        "https://api.weather.gov/points/44.05,-121.31",
        "https://api.weather.gov/points/45.52,-122.67",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("properties", {}).get("forecast")),
    weather,
)
forecast, fc_src, fc_live = fetch_first_json(
    [
        "https://api.open-meteo.com/v1/forecast?latitude=44.05&longitude=-121.31&hourly=temperature_2m&forecast_days=1",
        "https://api.open-meteo.com/v1/forecast?latitude=45.52&longitude=-122.67&hourly=temperature_2m&forecast_days=1",
    ],
    lambda d: isinstance(d, dict) and len(d.get("hourly", {}).get("temperature_2m", [])) > 0,
    forecast,
)

forest_count = len(forest.get("features", [])) if isinstance(forest, dict) else 0
if forest_count == 0 and isinstance(forest, dict) and forest.get("id"):
    forest_count = 1
print(f"Forestry records: {forest_count} | live={forest_live} | source={forest_src}")
print(f"NOAA forecast exists: {bool(weather.get('properties', {}).get('forecast'))} | live={wx_live} | source={wx_src}")
print(f"Open-Meteo hourly points: {len(forecast.get('hourly', {}).get('temperature_2m', []))} | live={fc_live} | source={fc_src}")


Forestry records: 1 | live=False | source=fallback
NOAA forecast exists: True | live=False | source=fallback
Open-Meteo hourly points: 1 | live=False | source=fallback


## Section B: Spatial Analysis (GeoPandas)


In [3]:
stands_data = {

    "stand_id": ["S1","S2","S3","S4","S5","S6"],

    "fuel_load": [0.72, 0.63, 0.81, 0.77, 0.58, 0.88],

    "slope":     [0.35, 0.25, 0.41, 0.39, 0.21, 0.45],

    "dist_km":   [0.8,  1.4,  0.6,  0.9,  1.8,  0.4],

    "zone":      ["A",  "A",  "B",  "B",  "A",  "B"],

}

lons = [-121.60, -121.20, -121.00, -120.80, -121.40, -120.95]

lats = [  44.30,   44.10,   44.40,   44.20,   43.90,   44.50]



gdf = gpd.GeoDataFrame(stands_data, geometry=gpd.points_from_xy(lons, lats), crs="EPSG:4326")

gdf["risk_score"] = (gdf["fuel_load"] * 0.5 + gdf["slope"] * 0.3

                     + (1.0 / gdf["dist_km"]) * 0.2).round(4)

gdf["risk_tier"] = gdf["risk_score"].apply(lambda r: "HIGH" if r > 0.75 else ("MED" if r > 0.55 else "LOW"))

gdf = gdf.sort_values("risk_score", ascending=False).reset_index(drop=True)

gdf["priority_rank"] = range(1, len(gdf) + 1)

print("Stand risk scores:")

print(gdf[["stand_id","risk_score","risk_tier","priority_rank"]].to_string(index=False))



# Fire station GeoDataFrame

stations_gdf = gpd.GeoDataFrame(

    {"station_id": ["FS1","FS2"]},

    geometry=gpd.points_from_xy([-121.35, -120.90], [44.20, 44.10]),

    crs="EPSG:4326",

)



# 1. Buffer: protection zones around stands (project to meters first)

gdf_proj = gdf.to_crs("EPSG:3857")

buf_zones = gdf_proj.buffer(8000).to_crs("EPSG:4326")

print(f"Buffer zones (8 km): {len(buf_zones)} polygons")



# 2. Nearest join: assign each stand to nearest fire station (km)

stands_m = gdf.to_crs("EPSG:3857")

stations_m = stations_gdf.to_crs("EPSG:3857")

nearest_sta_m = gpd.sjoin_nearest(stands_m, stations_m, how="left", distance_col="dist_m")

nearest_sta = nearest_sta_m.to_crs("EPSG:4326")

nearest_sta["dist_km"] = nearest_sta_m["dist_m"] / 1000.0

print("\nNearest fire station per stand (km):")

print(nearest_sta[["stand_id","station_id","dist_km"]].head(4).to_string(index=False))



# 3. Spatial join: stands within station buffer radius

sta_buf = stations_gdf.to_crs("EPSG:3857").copy()

sta_buf.geometry = sta_buf.buffer(50000).to_crs("EPSG:4326")  # 50km buffer

in_range = gpd.sjoin(gdf, sta_buf, how="inner", predicate="within")

print(f"\nStands within 50km of any fire station: {len(in_range)}")



# 4. Dissolve: aggregate by risk tier

dissolved = gdf.dissolve(by="risk_tier", aggfunc={"risk_score": "mean", "fuel_load": "mean"})

print("\nDissolved by risk tier:")

print(dissolved[["risk_score","fuel_load"]].to_string())



# 5. Bounding box filter: southern zone

south_gdf = gdf.cx[-121.65:-120.75, 43.85:44.15]

print(f"\nStands in southern zone: {list(south_gdf['stand_id'])}")



# 6. High risk overlay: clip to risk zone bbox

risk_bbox = box(-121.10, 44.10, -120.75, 44.55)

risk_mask = gpd.GeoDataFrame({"id": [1]}, geometry=[risk_bbox], crs="EPSG:4326")

clipped = gpd.clip(gdf, risk_mask)

print(f"Stands clipped to high-risk bbox: {list(clipped['stand_id'])}")



# 7. To file

gdf.to_file(str(OUTPUT_DIR / "d2-gpd-stands.geojson"), driver="GeoJSON")

print("\nWrote d2-gpd-stands.geojson")



# Inline visualization

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gdf.plot(ax=axes[0], column="risk_score", cmap="YlOrRd", markersize=140, legend=True,

         legend_kwds={"label": "Risk Score"})

for _, row in gdf.iterrows():

    axes[0].annotate(f"{row['stand_id']}\n#{row['priority_rank']}", (row.geometry.x, row.geometry.y),

                     textcoords="offset points", xytext=(4,4), fontsize=8)

stations_gdf.plot(ax=axes[0], color="blue", markersize=180, marker="^", label="Fire Station")

axes[0].set_title("Stand Risk Map (triangles=fire stations)")

axes[0].legend(); axes[0].grid(True, alpha=0.3)



gdf_s = gdf.sort_values("risk_score", ascending=False)

bar_colors = ["#c0392b" if t=="HIGH" else ("#e67e22" if t=="MED" else "#27ae60") for t in gdf_s["risk_tier"]]

axes[1].barh(gdf_s["stand_id"], gdf_s["risk_score"], color=bar_colors)

axes[1].set_xlabel("Fire Risk Score"); axes[1].set_title("Stand Priority Ranking")

axes[1].grid(True, axis="x", alpha=0.3)

plt.suptitle("D2 Forestry: GeoPandas Spatial Analysis", fontweight="bold")

plt.tight_layout(); plt.show()

# Basemap snapshot (real tiled basemap, saved as HTML)
try:
    import folium
    label_candidates = ["node", "stand_id", "asset_id", "zone_id"]
    label_col = next((c for c in label_candidates if c in gdf.columns), gdf.columns[0])
    center_lat = float(gdf.geometry.y.mean())
    center_lon = float(gdf.geometry.x.mean())
    fmap = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles="CartoDB positron")

    for _, row in gdf.iterrows():
        folium.CircleMarker(
            location=[float(row.geometry.y), float(row.geometry.x)],
            radius=6,
            color="#1d4ed8",
            fill=True,
            fill_opacity=0.85,
            popup=f"{label_col}: {row[label_col]}",
        ).add_to(fmap)

    if "demand_gdf" in locals():
        for _, row in demand_gdf.iterrows():
            folium.Marker(
                location=[float(row.geometry.y), float(row.geometry.x)],
                icon=folium.Icon(color="blue", icon="bolt", prefix="fa"),
                popup=f"demand_id: {row.get('demand_id', 'n/a')}",
            ).add_to(fmap)

    if "stations_gdf" in locals():
        for _, row in stations_gdf.iterrows():
            folium.Marker(
                location=[float(row.geometry.y), float(row.geometry.x)],
                icon=folium.Icon(color="green", icon="tree", prefix="fa"),
                popup=f"station_id: {row.get('station_id', 'n/a')}",
            ).add_to(fmap)

    if "incidents_gdf" in locals():
        for _, row in incidents_gdf.iterrows():
            folium.Marker(
                location=[float(row.geometry.y), float(row.geometry.x)],
                icon=folium.Icon(color="red", icon="warning-sign", prefix="glyphicon"),
                popup=f"inc_id: {row.get('inc_id', 'n/a')}",
            ).add_to(fmap)

    if "adapt_gdf" in locals():
        for _, row in adapt_gdf.iterrows():
            folium.Marker(
                location=[float(row.geometry.y), float(row.geometry.x)],
                icon=folium.Icon(color="purple", icon="leaf", prefix="fa"),
                popup=f"site_id: {row.get('site_id', 'n/a')}",
            ).add_to(fmap)

    map_path = OUTPUT_DIR / "d2-gpd-basemap.html"
    fmap.save(str(map_path))
    print(f"Wrote basemap snapshot: {map_path.name}")
    fmap
except Exception as exc:
    print(f"Basemap snapshot skipped: {exc}")


Stand risk scores:
stand_id  risk_score risk_tier  priority_rank
      S6      1.0750      HIGH              1
      S3      0.8613      HIGH              2
      S4      0.7242       MED              3
      S1      0.7150       MED              4
      S2      0.5329       LOW              5
      S5      0.4641       LOW              6
Buffer zones (8 km): 6 polygons

Nearest fire station per stand (km):
stand_id station_id   dist_km
      S6        FS2 62.465199
      S3        FS2 47.933287
      S4        FS2 19.095036
      S1        FS1 31.875074

Stands within 50km of any fire station: 7

Dissolved by risk tier:
           risk_score  fuel_load
risk_tier                       
HIGH          0.96815      0.845
LOW           0.49850      0.605
MED           0.71960      0.745

Stands in southern zone: ['S2', 'S5']
Stands clipped to high-risk bbox: ['S4', 'S3', 'S6']

Wrote d2-gpd-stands.geojson


C:\Users\Matt\AppData\Local\Temp\ipykernel_30192\3548832838.py:165: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Wrote basemap snapshot: d2-gpd-basemap.html


## Section C: Scenario Comparison


In [4]:
scenarios = {

    "no_action":       {"risk": 0.74, "cost_musd": 0.0,  "expected_loss_musd": 240.0},

    "thinning":        {"risk": 0.51, "cost_musd": 65.0, "expected_loss_musd": 170.0},

    "prescribed_burn": {"risk": 0.43, "cost_musd": 92.0, "expected_loss_musd": 135.0},

}

scenario_records = []

for name, vals in scenarios.items():

    score = round((1 - vals["risk"]) * 0.5 + (1.0 / max(vals["cost_musd"] + 1, 1)) * 10 * 0.2

                  + (1.0 / vals["expected_loss_musd"]) * 50 * 0.3, 4)

    scenario_records.append({"scenario": name, **vals, "score": score})

scenario_records.sort(key=lambda r: -float(r["score"]))



fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = [r["scenario"] for r in scenario_records]

colors = ["#27ae60", "#e67e22", "#c0392b"]

axes[0].barh(names, [r["risk"] for r in scenario_records], color=colors)

axes[0].set_xlabel("Fire Risk"); axes[0].set_title("Risk by Scenario"); axes[0].grid(True, axis="x", alpha=0.3)

axes[1].barh(names, [r["expected_loss_musd"] for r in scenario_records], color=colors)

axes[1].set_xlabel("Expected Loss ($M)"); axes[1].set_title("Expected Loss"); axes[1].grid(True, axis="x", alpha=0.3)

axes[2].barh(names, [r["score"] for r in scenario_records], color=colors)

axes[2].set_xlabel("Composite Score"); axes[2].set_title("Scenario Score"); axes[2].grid(True, axis="x", alpha=0.3)

plt.suptitle("D2 Forestry (GeoPandas): Scenario Comparison", fontweight="bold")

plt.tight_layout(); plt.show()



(OUTPUT_DIR / "d2-gpd-complex.json").write_text(

    json.dumps({"scenario_ranking": scenario_records}, indent=2, default=str), encoding="utf-8"

)

print("Wrote d2-gpd-complex.json")

for r in scenario_records:

    print(f"  {r['scenario']}: score={r['score']}")


Wrote d2-gpd-complex.json
  no_action: score=2.1925
  prescribed_burn: score=0.4176
  thinning: score=0.3635


C:\Users\Matt\AppData\Local\Temp\ipykernel_30192\2061752080.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
